## 1. Imports

In [11]:
import cv2
import numpy as np

## 2. Load trajectories

In [14]:
trajectories = np.load("trajectories_2d_segment.npy")
print(trajectories.shape)

(592, 58, 2)


## 3. Video settings

In [15]:
VIDEO_PATH = r"C:\projects\motion-capture-project\data\scene-01-take-01-cam-02.mp4"

cap = cv2.VideoCapture(VIDEO_PATH)

fps = cap.get(cv2.CAP_PROP_FPS)
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

cap.release()

print("FPS:", fps)
print("Size:", W, H)

FPS: 29.599269692399833
Size: 720 1280


## 4. Create video with dots and trails

In [16]:
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
OUTPUT_PATH = "../data/tracked_points_flow.mp4"

out = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (W, H))

num_frames, num_markers, _ = trajectories.shape

TRAIL_LENGTH = 20

for f in range(num_frames):
    frame = np.zeros((H, W, 3), dtype=np.uint8)

    for m in range(num_markers):
        x, y = trajectories[f, m]

        if np.isnan(x) or np.isnan(y):
            continue

        # dibujar estela
        start = max(0, f - TRAIL_LENGTH)

        for past_f in range(start, f):
            x1, y1 = trajectories[past_f, m]
            x2, y2 = trajectories[past_f + 1, m]

            if np.isnan(x1) or np.isnan(y1) or np.isnan(x2) or np.isnan(y2):
                continue

            cv2.line(
                frame,
                (int(x1), int(y1)),
                (int(x2), int(y2)),
                (0, 255, 0),
                2
            )

        # dibujar punto actual
        cv2.circle(
            frame,
            (int(x), int(y)),
            5,
            (0, 0, 255),
            -1
        )

        # dibujar ID del marcador
        cv2.putText(
            frame,
            str(m),
            (int(x) + 6, int(y) - 6),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.4,
            (255, 255, 255),
            1
        )

    out.write(frame)

out.release()

print("Video guardado:", OUTPUT_PATH)

Video guardado: ../data/tracked_points_flow.mp4


In [17]:
cap = cv2.VideoCapture(VIDEO_PATH)
OUTPUT_PATH = "../data/tracked_points_overlay.mp4"

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (W, H))

num_frames, num_markers, _ = trajectories.shape
TRAIL_LENGTH = 20

for f in range(num_frames):
    ok, frame = cap.read()
    if not ok:
        break

    for m in range(num_markers):
        x, y = trajectories[f, m]

        if np.isnan(x) or np.isnan(y):
            continue

        start = max(0, f - TRAIL_LENGTH)

        for past_f in range(start, f):
            x1, y1 = trajectories[past_f, m]
            x2, y2 = trajectories[past_f + 1, m]

            if np.isnan(x1) or np.isnan(y1) or np.isnan(x2) or np.isnan(y2):
                continue

            cv2.line(
                frame,
                (int(x1), int(y1)),
                (int(x2), int(y2)),
                (0, 255, 0),
                2
            )

        cv2.circle(frame, (int(x), int(y)), 5, (0, 0, 255), -1)

    out.write(frame)

cap.release()
out.release()

print("Video guardado: tracked_points_overlay.mp4")

Video guardado: tracked_points_overlay.mp4
